# How to Run Local LLMs with OpenAI Codex

## Setup Codex locally (Linux)

!apt update  
!sudo apt install nodejs npm -y  
!sudo npm install -g @openai/codex  (sudo when lack of permission)

* Note: That's it for the install - don't run codex yet. Running it bare drops you into OpenAI's "Sign in with ChatGPT" picker (which is modal - there's no escape hatch). Once we wire up a local profile, codex -p unsloth_api or codex -p llama_cpp skips that screen entirely because custom providers default to requires_openai_auth = false. Start the local model server first, then launch Codex against it.

## Unsloth Studio basic
* Install unsloth studio: curl -fsSL https://unsloth.ai/install.sh | sh
* unsloth studio -p 8888
* Test model locally
* Create unsloth API token: sk-unsloth-YOUR_TOKEN_HERE

### Connect Codex

#### Configure the Unsloth provider

web_search = "live"

[model_providers.unsloth_api]
name                  = "Unsloth Studio"
base_url              = "http://localhost:8888/v1"
env_key               = "UNSLOTH_STUDIO_AUTH_TOKEN"
wire_api              = "responses"
requires_openai_auth  = false

[profiles.unsloth_api]
model_provider = "unsloth_api"
model          = "unsloth/gemma-4-E4B-it-GGUF"

#### Set the API key env var

In [ ]:
# Put this line in the end of .bashrc or .zshrc (echo $SHELL)
export UNSLOTH_STUDIO_AUTH_TOKEN="sk-unsloth-YOUR_TOKEN_HERE"
# Verify:
source /.bashrc
echo $UNSLOTH_STUDIO_AUTH_TOKEN

### Start unsloth workspace

In [ ]:
# Create work space
mkdir unsloth-workspace && cd unsloth-workspace
# Launch Codex
codex -p unsloth_api

#### Handle error

* 1. Model metadata for `unsloth/gemma-4-E4B-it-GGUF` not found. Defaulting to fallback metadata; this can degrade performance and cause issues.
  + Add this to the top of ~/.codex/config.toml file:  
    model_context_window = 8192

### On start-up
* Open terminal 1 and:
  + unsloth studio -p 8888 => load model gemma-4-E4B-it-GGUF:UD-Q4_K_XL
* Open terminal 2:
  + codex -p unsloth_api --search --dangerously-bypass-approvals-and-sandbox
    - -p/-profile: select profile (unsloth_api in model_providers in config.toml)
    - --search equivalent web_search = "live" in config.toml
    - --search allow web search (Not working, on-hold) and --dangerously-bypass-approvals-and-sandbox bybass approval prompts

## Known issues:
* Even though docs says --search will enable the agent to search, but it says "I don't have general web search tool", config has set as https://developers.openai.com/codex/config-reference